In [6]:
using UUIDs

In [ ]:
abstract type AbstractDevice end
abstract type AbstractCPU <: AbstractDevice end
abstract type AbstractGPU <: AbstractDevice end
abstract type AbstractFPGA <: AbstractDevice end

# these will need more info like thread id or device id, numa region...
# maybe consider heterogeneous computing.jl
struct CPU <: AbstractCPU
    id::UUID
end
struct AMDGPU <: AbstractGPU
    id::UUID
end
struct ONEAPIGPU <: AbstractGPU
    id::UUID
end
struct AMDFPGA <: AbstractFPGA
    id::UUID
end

In [ ]:
# device endpoints carrying information how to send to this device
struct DeviceEndpoint{T <: AbstractDevice}
    id::UUID
end

# device manager with a type that indicates how sending to it works (via dispatch)
struct DeviceManager{T}
    # this device's endpoint
    this::DeviceEndpoint{T}

    # connections to other device managers to send to
    connections::Dict{UUID, DeviceEndpoint}
    
    # stash of received data that hasn't been requested yet
    stash::Dict{UUID, Any}
end

struct Machine
    devices::Dict{UUID, AbstractDevice}
    # ...
end

In [16]:
function create_manager(machine::Machine, dev::T_DEV) where {T_DEV <: AbstractDevice}
    # create an endpoint
    other_devices = Dict{UUID, DeviceEndpoint}()
    for (id, dev) in machine.devices
        @show dev
        other_devices[id] = DeviceEndpoint{typeof(dev)}(id)
    end

    return DeviceManager(
        DeviceEndpoint{T_DEV}(uuid1()),
        other_devices,
        Dict{UUID, Any}(),
    )
end

create_manager (generic function with 2 methods)

In [ ]:
"""
    get(manager::DeviceManager, id::UUID, T::Type)::T

Get a piece of data (identified by its id) that is expected to be 
transfered here (to the given DeviceManager). If it is not yet there it will block
before returning, until it has arrived.
`T` is the return type of the function and type of the expected data.
"""
function get end


"""
    send(manager::DeviceManager, target::UUID, payload::T, payload_id::UUID)::Nothing

Send a piece of data `payload` with its identifying id `payload_id` to the target device, identified
by its `target` ID.
"""
function send end

In [18]:
devices = Dict{UUID, AbstractDevice}()

cpu_id = UUIDs.uuid1()
devices[cpu_id] = CPU(cpu_id)

gpu_id = UUIDs.uuid1()
devices[gpu_id] = AMDGPU(gpu_id)

fpga_id = UUIDs.uuid1()
devices[fpga_id] = AMDFPGA(fpga_id)

machine = Machine(devices)

@show cpu_manager = create_manager(machine, devices[cpu_id])

Base.summarysize(cpu_manager)

dev = AMDFPGA(UUID("eac513ba-23b1-11f1-8b6b-f53d97062b11"))
dev = CPU(UUID("eac4ff4a-23b1-11f1-81ac-49ca0e03f58a"))
dev = AMDGPU(UUID("eac5092e-23b1-11f1-b871-adc70f022779"))
cpu_manager = create_manager(machine, devices[cpu_id]) = DeviceManager{CPU}(DeviceEndpoint{CPU}(UUID("eac52bde-23b1-11f1-b29e-61c8b65089fa")), Dict{UUID, DeviceEndpoint}(UUID("eac513ba-23b1-11f1-8b6b-f53d97062b11") => DeviceEndpoint{AMDFPGA}(UUID("eac513ba-23b1-11f1-8b6b-f53d97062b11")), UUID("eac4ff4a-23b1-11f1-81ac-49ca0e03f58a") => DeviceEndpoint{CPU}(UUID("eac4ff4a-23b1-11f1-81ac-49ca0e03f58a")), UUID("eac5092e-23b1-11f1-b871-adc70f022779") => DeviceEndpoint{AMDGPU}(UUID("eac5092e-23b1-11f1-b871-adc70f022779"))), Dict{UUID, Any}())


704

One-Sided Communication in MPI:
- https://enccs.github.io/intermediate-mpi/one-sided-concepts/
- https://juliaparallel.org/MPI.jl/stable/reference/onesided/

Communication in ZMQ:
- see example/